In [79]:
!pip install pytorch-crf -q

In [80]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

SEED = 42

EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

WORD_EMBEDDING_DIM = 100
CHAR_EMBEDDING_DIM = 30
CHAR_CHANNELS = 50
HIDDEN_DIM = 128
DROPOUT = 0.3

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [81]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)

Seed: 42


In [82]:
datasets = {
    "Barishal": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Barishal_NER.csv"),
    "Chittagong": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Chittagong_NER.csv"),
    "Mymensingh": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Mymensingh_NER.csv"),
    "Noakhali": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Noakhali_NER.csv"),
    "Sylhet": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Sylhet_NER.csv")
}

print("Datasets loaded.")

Datasets loaded.


In [83]:
sentence_data = {}

for dialect, df in datasets.items():

    df["Sentence #"] = df["Sentence #"].ffill()

    word_column = [
        c for c in df.columns
        if c.endswith("_word")
    ][0]

    sentences = []

    for sentence_id, group in df.groupby("Sentence #"):

        sentences.append({
            "sentence_id": int(
                str(sentence_id).replace("Sentence: ", "")
            ),
            "tokens": group[word_column].astype(str).tolist(),
            "tags": group["bio_tag"].astype(str).tolist()
        })

    sentence_data[dialect] = sentences

for dialect in sentence_data:
    print(
        dialect,
        "| Sentences:",
        len(sentence_data[dialect])
    )

Barishal | Sentences: 3481
Chittagong | Sentences: 3481
Mymensingh | Sentences: 3481
Noakhali | Sentences: 3481
Sylhet | Sentences: 3481


In [84]:
tags = set()

for sentences in sentence_data.values():

    for sentence in sentences:
        tags.update(sentence["tags"])

tags = sorted(tags)

tag_to_id = {
    tag: i
    for i, tag in enumerate(tags)
}

id_to_tag = {
    i: tag
    for tag, i in tag_to_id.items()
}

print("Number of tags:", len(tags))
print(tag_to_id)

Number of tags: 20
{'\nI-LOC': 0, 'B-ANI': 1, 'B-COL': 2, 'B-FOOD': 3, 'B-LOC': 4, 'B-OBJ': 5, 'B-ORG': 6, 'B-PER': 7, 'B-REL': 8, 'B-ROLE': 9, 'I-ANI': 10, 'I-COL': 11, 'I-FOOD': 12, 'I-LOC': 13, 'I-OBJ': 14, 'I-ORG': 15, 'I-PER': 16, 'I-REL': 17, 'I-ROLE': 18, 'O': 19}


In [85]:
def get_cross_dialect_data(held_out):

    train_sentences = []
    test_sentences = sentence_data[held_out]

    for dialect in sentence_data:

        if dialect != held_out:
            train_sentences.extend(
                sentence_data[dialect]
            )

    return train_sentences, test_sentences

In [86]:
def build_word_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sentence in sentences:

        for token in sentence["tokens"]:

            if token not in vocab:
                vocab[token] = len(vocab)

    return vocab

In [87]:
def build_char_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sentence in sentences:

        for word in sentence["tokens"]:

            for char in word:

                if char not in vocab:
                    vocab[char] = len(vocab)

    return vocab

In [88]:
def encode_sentence(sentence, vocab):

    word_ids = [
        vocab.get(
            token,
            vocab["<UNK>"]
        )
        for token in sentence["tokens"]
    ]

    tag_ids = [
        tag_to_id[tag]
        for tag in sentence["tags"]
    ]

    return word_ids, tag_ids


def encode_characters(sentence, char_vocab):

    char_ids = []

    for word in sentence["tokens"]:

        chars = [
            char_vocab.get(
                char,
                char_vocab["<UNK>"]
            )
            for char in word
        ]

        char_ids.append(chars)

    return char_ids

In [89]:
class NERDataset(Dataset):

    def __init__(self, sentences, vocab):

        self.data = [
            encode_sentence(sentence, vocab)
            for sentence in sentences
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        words, tags = self.data[index]

        return (
            torch.tensor(words, dtype=torch.long),
            torch.tensor(tags, dtype=torch.long)
        )

In [90]:
def collate_fn(batch):

    words, tags = zip(*batch)

    max_length = max(
        len(x) for x in words
    )

    padded_words = []
    padded_tags = []
    masks = []

    for word_ids, tag_ids in zip(words, tags):

        padding = max_length - len(word_ids)

        padded_words.append(
            torch.cat([
                word_ids,
                torch.zeros(
                    padding,
                    dtype=torch.long
                )
            ])
        )

        padded_tags.append(
            torch.cat([
                tag_ids,
                torch.zeros(
                    padding,
                    dtype=torch.long
                )
            ])
        )

        masks.append(
            torch.tensor(
                [1] * len(word_ids) +
                [0] * padding,
                dtype=torch.bool
            )
        )

    return (
        torch.stack(padded_words),
        torch.stack(padded_tags),
        torch.stack(masks)
    )

In [91]:
class CharNERDataset(Dataset):

    def __init__(
        self,
        sentences,
        word_vocab,
        char_vocab
    ):

        self.data = []

        for sentence in sentences:

            word_ids, tag_ids = encode_sentence(
                sentence,
                word_vocab
            )

            char_ids = encode_characters(
                sentence,
                char_vocab
            )

            self.data.append(
                (
                    word_ids,
                    char_ids,
                    tag_ids
                )
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        return self.data[index]

In [92]:
def char_collate_fn(batch):

    words, chars, tags = zip(*batch)

    max_sentence_length = max(
        len(x) for x in words
    )

    max_word_length = max(
        len(char)
        for sentence in chars
        for char in sentence
    )

    padded_words = []
    padded_tags = []
    padded_chars = []
    masks = []

    for word_ids, char_ids, tag_ids in zip(
        words,
        chars,
        tags
    ):

        sentence_length = len(word_ids)
        sentence_padding = (
            max_sentence_length -
            sentence_length
        )

        padded_words.append(
            word_ids +
            [0] * sentence_padding
        )

        padded_tags.append(
            tag_ids +
            [0] * sentence_padding
        )

        masks.append(
            [1] * sentence_length +
            [0] * sentence_padding
        )

        sentence_chars = []

        for word in char_ids:

            word_padding = (
                max_word_length -
                len(word)
            )

            sentence_chars.append(
                word +
                [0] * word_padding
            )

        for _ in range(sentence_padding):

            sentence_chars.append(
                [0] * max_word_length
            )

        padded_chars.append(sentence_chars)

    return (
        torch.tensor(
            padded_words,
            dtype=torch.long
        ),
        torch.tensor(
            padded_chars,
            dtype=torch.long
        ),
        torch.tensor(
            padded_tags,
            dtype=torch.long
        ),
        torch.tensor(
            masks,
            dtype=torch.bool
        )
    )

In [93]:
class BiLSTMCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        tag_size
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            WORD_EMBEDDING_DIM,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=WORD_EMBEDDING_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.fc = nn.Linear(
            HIDDEN_DIM * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(
        self,
        tokens,
        tags=None,
        mask=None
    ):

        x = self.embedding(tokens)

        x, _ = self.lstm(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            return -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(
            emissions,
            mask=mask
        )

In [94]:
class BiGRUCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        tag_size
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            WORD_EMBEDDING_DIM,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=WORD_EMBEDDING_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.fc = nn.Linear(
            HIDDEN_DIM * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(
        self,
        tokens,
        tags=None,
        mask=None
    ):

        x = self.embedding(tokens)

        x, _ = self.gru(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            return -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(
            emissions,
            mask=mask
        )

In [95]:
class CharCNNBiLSTMCRF(nn.Module):

    def __init__(
        self,
        word_vocab_size,
        char_vocab_size,
        tag_size
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            word_vocab_size,
            WORD_EMBEDDING_DIM,
            padding_idx=0
        )

        self.char_embedding = nn.Embedding(
            char_vocab_size,
            CHAR_EMBEDDING_DIM,
            padding_idx=0
        )

        self.char_cnn = nn.Conv1d(
            in_channels=CHAR_EMBEDDING_DIM,
            out_channels=CHAR_CHANNELS,
            kernel_size=3,
            padding=1
        )

        self.lstm = nn.LSTM(
            input_size=
                WORD_EMBEDDING_DIM +
                CHAR_CHANNELS,
            hidden_size=HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.fc = nn.Linear(
            HIDDEN_DIM * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(
        self,
        words,
        chars,
        tags=None,
        mask=None
    ):

        word_emb = self.word_embedding(words)

        char_emb = self.char_embedding(chars)

        batch_size = chars.size(0)
        sentence_length = chars.size(1)
        word_length = chars.size(2)

        char_emb = char_emb.view(
            batch_size * sentence_length,
            word_length,
            CHAR_EMBEDDING_DIM
        )

        char_emb = char_emb.transpose(1, 2)

        char_features = self.char_cnn(
            char_emb
        )

        char_features = torch.max(
            char_features,
            dim=2
        ).values

        char_features = char_features.view(
            batch_size,
            sentence_length,
            CHAR_CHANNELS
        )

        x = torch.cat(
            [
                word_emb,
                char_features
            ],
            dim=2
        )

        x, _ = self.lstm(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            return -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(
            emissions,
            mask=mask
        )

In [96]:
def get_entities(tags):

    entities = []

    start = None
    entity_type = None

    for i, tag in enumerate(tags + ["O"]):

        if tag.startswith("B-"):

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = i
            entity_type = tag[2:]

        elif tag.startswith("I-"):

            if start is None:
                start = i
                entity_type = tag[2:]

        else:

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = None
            entity_type = None

    return set(entities)

In [97]:
def train_word_model(
    model,
    loader
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for words, tags, mask in loader:

            words = words.to(device)
            tags = tags.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            loss = model(
                words,
                tags=tags,
                mask=mask
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = (
            total_loss /
            len(loader)
        )

        print(
            f"Epoch {epoch + 1}/{EPOCHS}"
            f" - Loss: {avg_loss:.4f}"
        )

In [98]:
def train_char_model(
    model,
    loader
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for words, chars, tags, mask in loader:

            words = words.to(device)
            chars = chars.to(device)
            tags = tags.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            loss = model(
                words,
                chars,
                tags=tags,
                mask=mask
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = (
            total_loss /
            len(loader)
        )

        print(
            f"Epoch {epoch + 1}/{EPOCHS}"
            f" - Loss: {avg_loss:.4f}"
        )

In [99]:
def evaluate_word_model(
    model,
    loader
):

    model.eval()

    correct = 0
    predicted = 0
    actual = 0

    with torch.no_grad():

        for words, tags, mask in loader:

            words = words.to(device)
            mask = mask.to(device)

            predictions = model(
                words,
                mask=mask
            )

            for i, pred in enumerate(
                predictions
            ):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                true_entities = get_entities(
                    true_tags
                )

                pred_entities = get_entities(
                    pred_tags
                )

                correct += len(
                    true_entities &
                    pred_entities
                )

                actual += len(true_entities)
                predicted += len(pred_entities)

    precision = (
        correct / predicted
        if predicted else 0
    )

    recall = (
        correct / actual
        if actual else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall else 0
    )

    return precision, recall, f1

In [100]:
def evaluate_char_model(
    model,
    loader
):

    model.eval()

    correct = 0
    predicted = 0
    actual = 0

    with torch.no_grad():

        for words, chars, tags, mask in loader:

            words = words.to(device)
            chars = chars.to(device)
            mask = mask.to(device)

            predictions = model(
                words,
                chars,
                mask=mask
            )

            for i, pred in enumerate(
                predictions
            ):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                true_entities = get_entities(
                    true_tags
                )

                pred_entities = get_entities(
                    pred_tags
                )

                correct += len(
                    true_entities &
                    pred_entities
                )

                actual += len(true_entities)
                predicted += len(pred_entities)

    precision = (
        correct / predicted
        if predicted else 0
    )

    recall = (
        correct / actual
        if actual else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall else 0
    )

    return precision, recall, f1

In [101]:
all_results = []

dialects = list(sentence_data.keys())

for held_out in dialects:

    print("\n")
    print("=" * 70)
    print("HELD-OUT DIALECT:", held_out)
    print("=" * 70)

    # Create train/test data

    train_sentences, test_sentences = (
        get_cross_dialect_data(held_out)
    )

    print(
        "Training sentences:",
        len(train_sentences)
    )

    print(
        "Test sentences:",
        len(test_sentences)
    )

    # Build training-only vocabularies

    word_vocab = build_word_vocab(
        train_sentences
    )

    char_vocab = build_char_vocab(
        train_sentences
    )

    print(
        "Word vocabulary:",
        len(word_vocab)
    )

    print(
        "Character vocabulary:",
        len(char_vocab)
    )


    # Word datasets/loaders


    train_dataset = NERDataset(
        train_sentences,
        word_vocab
    )

    test_dataset = NERDataset(
        test_sentences,
        word_vocab
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )


    # Character datasets/loaders

    char_train_dataset = CharNERDataset(
        train_sentences,
        word_vocab,
        char_vocab
    )

    char_test_dataset = CharNERDataset(
        test_sentences,
        word_vocab,
        char_vocab
    )

    char_train_loader = DataLoader(
        char_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=char_collate_fn
    )

    char_test_loader = DataLoader(
        char_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=char_collate_fn
    )

    # MODEL 1: BiLSTM-CRF

    print("\n--- BiLSTM-CRF ---")

    model = BiLSTMCRF(
        vocab_size=len(word_vocab),
        tag_size=len(tag_to_id)
    ).to(device)

    train_word_model(
        model,
        train_loader
    )

    precision, recall, f1 = (
        evaluate_word_model(
            model,
            test_loader
        )
    )

    all_results.append({
        "Held-out Dialect": held_out,
        "Model": "BiLSTM-CRF",
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    print(
        f"F1: {f1:.4f}"
    )


    # MODEL 2: BiGRU-CRF

    print("\n--- BiGRU-CRF ---")

    model = BiGRUCRF(
        vocab_size=len(word_vocab),
        tag_size=len(tag_to_id)
    ).to(device)

    train_word_model(
        model,
        train_loader
    )

    precision, recall, f1 = (
        evaluate_word_model(
            model,
            test_loader
        )
    )

    all_results.append({
        "Held-out Dialect": held_out,
        "Model": "BiGRU-CRF",
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    print(
        f"F1: {f1:.4f}"
    )


    # MODEL 3: Char-CNN-BiLSTM-CRF

    print("\n--- Char-CNN-BiLSTM-CRF ---")

    model = CharCNNBiLSTMCRF(
        word_vocab_size=len(word_vocab),
        char_vocab_size=len(char_vocab),
        tag_size=len(tag_to_id)
    ).to(device)

    train_char_model(
        model,
        char_train_loader
    )

    precision, recall, f1 = (
        evaluate_char_model(
            model,
            char_test_loader
        )
    )

    all_results.append({
        "Held-out Dialect": held_out,
        "Model": "Char-CNN-BiLSTM-CRF",
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    print(
        f"F1: {f1:.4f}"
    )



HELD-OUT DIALECT: Barishal
Training sentences: 13924
Test sentences: 3481
Word vocabulary: 12305
Character vocabulary: 88

--- BiLSTM-CRF ---
Epoch 1/20 - Loss: 3.3989
Epoch 2/20 - Loss: 1.6710
Epoch 3/20 - Loss: 1.0794
Epoch 4/20 - Loss: 0.7103
Epoch 5/20 - Loss: 0.4636
Epoch 6/20 - Loss: 0.2993
Epoch 7/20 - Loss: 0.1881
Epoch 8/20 - Loss: 0.1257
Epoch 9/20 - Loss: 0.0865
Epoch 10/20 - Loss: 0.0642
Epoch 11/20 - Loss: 0.0477
Epoch 12/20 - Loss: 0.0390
Epoch 13/20 - Loss: 0.0355
Epoch 14/20 - Loss: 0.0318
Epoch 15/20 - Loss: 0.0282
Epoch 16/20 - Loss: 0.0229
Epoch 17/20 - Loss: 0.0214
Epoch 18/20 - Loss: 0.0166
Epoch 19/20 - Loss: 0.0152
Epoch 20/20 - Loss: 0.0154
F1: 0.3883

--- BiGRU-CRF ---
Epoch 1/20 - Loss: 3.2323
Epoch 2/20 - Loss: 1.6062
Epoch 3/20 - Loss: 1.0653
Epoch 4/20 - Loss: 0.7045
Epoch 5/20 - Loss: 0.4579
Epoch 6/20 - Loss: 0.2934
Epoch 7/20 - Loss: 0.1891
Epoch 8/20 - Loss: 0.1275
Epoch 9/20 - Loss: 0.0868
Epoch 10/20 - Loss: 0.0640
Epoch 11/20 - Loss: 0.0520
Epoch 1

In [102]:
cross_dialect_results = pd.DataFrame(
    all_results
)

cross_dialect_results

,Held-out Dialect,Model,Precision,Recall,F1
0,Barishal,BiLSTM-CRF,0.282463,0.621180,0.388340
1,Barishal,BiGRU-CRF,0.217447,0.623815,0.322484
2,Barishal,Char-CNN-BiLSTM-CRF,0.574793,0.732877,0.644280
3,Chittagong,BiLSTM-CRF,0.169644,0.559151,0.260311
4,Chittagong,BiGRU-CRF,0.777778,0.497613,0.606923
5,Chittagong,Char-CNN-BiLSTM-CRF,0.800898,0.567639,0.664390
6,Mymensingh,BiLSTM-CRF,0.887364,0.824217,0.854626
7,Mymensingh,BiGRU-CRF,0.719434,0.837493,0.773988
8,Mymensingh,Char-CNN-BiLSTM-CRF,0.872940,0.872013,0.872476
9,Noakhali,BiLSTM-CRF,0.828086,0.699096,0.758144


In [103]:
summary = (
    cross_dialect_results
    .groupby("Model")["F1"]
    .agg(["mean", "std"])
    .reset_index()
)

summary

,Model,mean,std
0,BiGRU-CRF,0.641474,0.190359
1,BiLSTM-CRF,0.605809,0.263610
2,Char-CNN-BiLSTM-CRF,0.741440,0.093828


In [104]:
cross_dialect_results.to_csv(
    "/kaggle/working/cross_dialect_results.csv",
    index=False
)

summary.to_csv(
    "/kaggle/working/cross_dialect_summary.csv",
    index=False
)

print("Results saved.")

Results saved.


In [105]:
summary = (
    cross_dialect_results
    .groupby("Model")[["Precision", "Recall", "F1"]]
    .agg(["mean", "std"])
)

summary

Precision              Recall                  F1  \
                         mean       std      mean       std      mean   
Model                                                                   
BiGRU-CRF            0.673043  0.258381  0.668047  0.123246  0.641474   
BiLSTM-CRF           0.604663  0.348547  0.679919  0.099350  0.605809   
Char-CNN-BiLSTM-CRF  0.765746  0.114919  0.729640  0.109387  0.741440   

                               
                          std  
Model                          
BiGRU-CRF            0.190359  
BiLSTM-CRF           0.263610  
Char-CNN-BiLSTM-CRF  0.093828

In [106]:
model_means = (
    cross_dialect_results
    .groupby("Model")["F1"]
    .mean()
    .sort_values(ascending=False)
)

model_means

Model
Char-CNN-BiLSTM-CRF    0.741440
BiGRU-CRF              0.641474
BiLSTM-CRF             0.605809
Name: F1, dtype: float64

In [107]:
best_models = (
    cross_dialect_results
    .loc[
        cross_dialect_results
        .groupby("Held-out Dialect")["F1"]
        .idxmax()
    ]
    .sort_values("Held-out Dialect")
)

best_models[
    ["Held-out Dialect", "Model", "F1"]
]

,Held-out Dialect,Model,F1
2,Barishal,Char-CNN-BiLSTM-CRF,0.644280
5,Chittagong,Char-CNN-BiLSTM-CRF,0.664390
8,Mymensingh,Char-CNN-BiLSTM-CRF,0.872476
11,Noakhali,Char-CNN-BiLSTM-CRF,0.792484
12,Sylhet,BiLSTM-CRF,0.767624


In [108]:
pivot = cross_dialect_results.pivot(
    index="Held-out Dialect",
    columns="Model",
    values="F1"
)

pivot["CharCNN - BiLSTM"] = (
    pivot["Char-CNN-BiLSTM-CRF"]
    - pivot["BiLSTM-CRF"]
)

pivot

Model,BiGRU-CRF,BiLSTM-CRF,Char-CNN-BiLSTM-CRF,CharCNN - BiLSTM
Held-out Dialect,,,,
Barishal,0.322484,0.388340,0.644280,0.255940
Chittagong,0.606923,0.260311,0.664390,0.404079
Mymensingh,0.773988,0.854626,0.872476,0.017851
Noakhali,0.754335,0.758144,0.792484,0.034341
Sylhet,0.749642,0.767624,0.733568,-0.034056


In [109]:
pivot["CharCNN - BiGRU"] = (
    pivot["Char-CNN-BiLSTM-CRF"]
    - pivot["BiGRU-CRF"]
)

pivot

Model,BiGRU-CRF,BiLSTM-CRF,Char-CNN-BiLSTM-CRF,CharCNN - BiLSTM,CharCNN - BiGRU
Held-out Dialect,,,,,
Barishal,0.322484,0.388340,0.644280,0.255940,0.321796
Chittagong,0.606923,0.260311,0.664390,0.404079,0.057467
Mymensingh,0.773988,0.854626,0.872476,0.017851,0.098488
Noakhali,0.754335,0.758144,0.792484,0.034341,0.038149
Sylhet,0.749642,0.767624,0.733568,-0.034056,-0.016074


In [110]:
variation = (
    cross_dialect_results
    .groupby("Model")["F1"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

variation

,mean,std,min,max
Model,,,,
Char-CNN-BiLSTM-CRF,0.741440,0.093828,0.644280,0.872476
BiGRU-CRF,0.641474,0.190359,0.322484,0.773988
BiLSTM-CRF,0.605809,0.263610,0.260311,0.854626


In [111]:
dialect_difficulty = (
    cross_dialect_results
    .groupby("Held-out Dialect")["F1"]
    .mean()
    .sort_values()
)

dialect_difficulty

Held-out Dialect
Barishal      0.451701
Chittagong    0.510541
Sylhet        0.750278
Noakhali      0.768321
Mymensingh    0.833696
Name: F1, dtype: float64

In [112]:
differences = pd.DataFrame({
    "Dialect": pivot.index,
    "CharCNN_vs_BiLSTM": pivot["CharCNN - BiLSTM"],
    "CharCNN_vs_BiGRU": pivot["CharCNN - BiGRU"]
})

differences

,Dialect,CharCNN_vs_BiLSTM,CharCNN_vs_BiGRU
Held-out Dialect,,,
Barishal,Barishal,0.255940,0.321796
Chittagong,Chittagong,0.404079,0.057467
Mymensingh,Mymensingh,0.017851,0.098488
Noakhali,Noakhali,0.034341,0.038149
Sylhet,Sylhet,-0.034056,-0.016074


In [113]:
print(
    "Char-CNN vs BiLSTM mean improvement:",
    differences["CharCNN_vs_BiLSTM"].mean()
)

print(
    "Char-CNN vs BiGRU mean improvement:",
    differences["CharCNN_vs_BiGRU"].mean()
)

Char-CNN vs BiLSTM mean improvement: 0.13563068420701946
Char-CNN vs BiGRU mean improvement: 0.0999651637413865


In [114]:
pivot.to_csv(
    "cross_dialect_model_comparison.csv"
)

variation.to_csv(
    "cross_dialect_model_variation.csv"
)

dialect_difficulty.to_csv(
    "cross_dialect_dialect_difficulty.csv"
)

print("Analysis tables saved.")

Analysis tables saved.


In [115]:
vocab_coverage = []

for held_out in sentence_data:

    train_sentences, test_sentences = get_cross_dialect_data(
        held_out
    )

    train_vocab = set()

    for sentence in train_sentences:
        train_vocab.update(sentence["tokens"])

    test_vocab = set()

    for sentence in test_sentences:
        test_vocab.update(sentence["tokens"])

    covered = test_vocab & train_vocab

    coverage = len(covered) / len(test_vocab)

    vocab_coverage.append({
        "Held-out Dialect": held_out,
        "Test Vocabulary": len(test_vocab),
        "Training Vocabulary": len(train_vocab),
        "Covered Words": len(covered),
        "Vocabulary Coverage": coverage
    })

vocab_coverage = pd.DataFrame(vocab_coverage)

vocab_coverage

,Held-out Dialect,Test Vocabulary,Training Vocabulary,Covered Words,Vocabulary Coverage
0,Barishal,3805,12303,2213,0.581603
1,Chittagong,4942,10719,1766,0.357345
2,Mymensingh,3883,12726,2714,0.698944
3,Noakhali,4592,11858,2555,0.556402
4,Sylhet,4345,11737,2187,0.503337


In [116]:
entity_shift = []

for held_out in sentence_data:

    train_sentences, test_sentences = get_cross_dialect_data(
        held_out
    )

    train_counts = {}
    test_counts = {}

    # Training entity counts
    for sentence in train_sentences:

        entities = get_entities(sentence["tags"])

        for _, _, entity_type in entities:
            train_counts[entity_type] = (
                train_counts.get(entity_type, 0) + 1
            )

    # Test entity counts
    for sentence in test_sentences:

        entities = get_entities(sentence["tags"])

        for _, _, entity_type in entities:
            test_counts[entity_type] = (
                test_counts.get(entity_type, 0) + 1
            )

    entity_types = set(train_counts) | set(test_counts)

    train_total = sum(train_counts.values())
    test_total = sum(test_counts.values())

    differences = []

    for entity_type in entity_types:

        train_prop = (
            train_counts.get(entity_type, 0)
            / train_total
        )

        test_prop = (
            test_counts.get(entity_type, 0)
            / test_total
        )

        differences.append(
            abs(train_prop - test_prop)
        )

    entity_shift.append({
        "Held-out Dialect": held_out,
        "Entity Distribution Shift": np.mean(
            differences
        )
    })

entity_shift = pd.DataFrame(entity_shift)

entity_shift

,Held-out Dialect,Entity Distribution Shift
0,Barishal,0.000537
1,Chittagong,0.001085
2,Mymensingh,0.000769
3,Noakhali,0.000977
4,Sylhet,0.001029


In [117]:
entity_distribution = []

for held_out in sentence_data:

    train_sentences, test_sentences = get_cross_dialect_data(
        held_out
    )

    train_counts = {}
    test_counts = {}

    for sentence in train_sentences:

        for _, _, entity_type in get_entities(sentence["tags"]):
            train_counts[entity_type] = (
                train_counts.get(entity_type, 0) + 1
            )

    for sentence in test_sentences:

        for _, _, entity_type in get_entities(sentence["tags"]):
            test_counts[entity_type] = (
                test_counts.get(entity_type, 0) + 1
            )

    train_total = sum(train_counts.values())
    test_total = sum(test_counts.values())

    for entity_type in sorted(
        set(train_counts) | set(test_counts)
    ):

        train_prop = (
            train_counts.get(entity_type, 0)
            / train_total
        )

        test_prop = (
            test_counts.get(entity_type, 0)
            / test_total
        )

        entity_distribution.append({
            "Held-out Dialect": held_out,
            "Entity Type": entity_type,
            "Train Proportion": train_prop,
            "Test Proportion": test_prop
        })

entity_distribution = pd.DataFrame(
    entity_distribution
)

entity_distribution

,Held-out Dialect,Entity Type,Train Proportion,Test Proportion
0,Barishal,ANI,0.025960,0.025817
1,Barishal,COL,0.062119,0.062698
2,Barishal,FOOD,0.136954,0.136986
3,Barishal,LOC,0.165960,0.164910
4,Barishal,OBJ,0.171788,0.172813
5,Barishal,ORG,0.059603,0.058483
6,Barishal,PER,0.018543,0.018440
7,Barishal,REL,0.316026,0.316122
8,Barishal,ROLE,0.043046,0.043730
9,Chittagong,ANI,0.025916,0.025995


--------------------------------------------------------------------------

---------------------------------------------------

In [118]:
char_error_results = {}

In [119]:
for held_out in dialects:

    print("\n" + "=" * 60)
    print("Error Analysis:", held_out)
    print("=" * 60)

    train_sentences, test_sentences = get_cross_dialect_data(
        held_out
    )

    # Training-only vocabularies
    word_vocab = build_word_vocab(train_sentences)
    char_vocab = build_char_vocab(train_sentences)

    # Datasets
    train_dataset = CharNERDataset(
        train_sentences,
        word_vocab,
        char_vocab
    )

    test_dataset = CharNERDataset(
        test_sentences,
        word_vocab,
        char_vocab
    )

    # Loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=char_collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=char_collate_fn
    )

    # Model
    model = CharCNNBiLSTMCRF(
        word_vocab_size=len(word_vocab),
        char_vocab_size=len(char_vocab),
        tag_size=len(tag_to_id)
    ).to(device)

    # Same training protocol
    train_char_model(
        model,
        train_loader
    )

    # Save predictions
    model.eval()

    predictions = []

    with torch.no_grad():

        for batch in test_loader:

            words, chars, tags, mask = batch

            words = words.to(device)
            chars = chars.to(device)
            mask = mask.to(device)

            batch_predictions = model(
                words,
                chars,
                mask=mask
            )

            for i, pred in enumerate(batch_predictions):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                predictions.append({
                    "true_tags": true_tags,
                    "pred_tags": pred_tags
                })

    char_error_results[held_out] = {
        "sentences": test_sentences,
        "predictions": predictions
    }

    print(
        "Saved predictions:",
        len(predictions)
    )


Error Analysis: Barishal
Epoch 1/20 - Loss: 2.8276
Epoch 2/20 - Loss: 1.2996
Epoch 3/20 - Loss: 0.8036
Epoch 4/20 - Loss: 0.5243
Epoch 5/20 - Loss: 0.3454
Epoch 6/20 - Loss: 0.2199
Epoch 7/20 - Loss: 0.1483
Epoch 8/20 - Loss: 0.1002
Epoch 9/20 - Loss: 0.0725
Epoch 10/20 - Loss: 0.0564
Epoch 11/20 - Loss: 0.0492
Epoch 12/20 - Loss: 0.0387
Epoch 13/20 - Loss: 0.0324
Epoch 14/20 - Loss: 0.0264
Epoch 15/20 - Loss: 0.0246
Epoch 16/20 - Loss: 0.0324
Epoch 17/20 - Loss: 0.0236
Epoch 18/20 - Loss: 0.0195
Epoch 19/20 - Loss: 0.0157
Epoch 20/20 - Loss: 0.0171
Saved predictions: 3481

Error Analysis: Chittagong
Epoch 1/20 - Loss: 2.7882
Epoch 2/20 - Loss: 1.1941
Epoch 3/20 - Loss: 0.7175
Epoch 4/20 - Loss: 0.4531
Epoch 5/20 - Loss: 0.2955
Epoch 6/20 - Loss: 0.1875
Epoch 7/20 - Loss: 0.1191
Epoch 8/20 - Loss: 0.0846
Epoch 9/20 - Loss: 0.0615
Epoch 10/20 - Loss: 0.0507
Epoch 11/20 - Loss: 0.0487
Epoch 12/20 - Loss: 0.0332
Epoch 13/20 - Loss: 0.0265
Epoch 14/20 - Loss: 0.0246
Epoch 15/20 - Loss: 0.

In [120]:
for dialect in char_error_results:

    print(
        dialect,
        "→",
        len(char_error_results[dialect]["predictions"]),
        "predictions"
    )

Barishal → 3481 predictions
Chittagong → 3481 predictions
Mymensingh → 3481 predictions
Noakhali → 3481 predictions
Sylhet → 3481 predictions


In [121]:
from collections import defaultdict

entity_results = []

for dialect in char_error_results:

    type_correct = defaultdict(int)
    type_actual = defaultdict(int)

    predictions = char_error_results[dialect]["predictions"]

    for item in predictions:

        true_entities = get_entities(item["true_tags"])
        pred_entities = get_entities(item["pred_tags"])

        for start, end, entity_type in true_entities:

            type_actual[entity_type] += 1

            if (start, end, entity_type) in pred_entities:
                type_correct[entity_type] += 1

    for entity_type in sorted(type_actual):

        recall = (
            type_correct[entity_type]
            / type_actual[entity_type]
        )

        entity_results.append({
            "Dialect": dialect,
            "Entity Type": entity_type,
            "Gold": type_actual[entity_type],
            "Correct": type_correct[entity_type],
            "Recall": recall
        })

entity_results = pd.DataFrame(entity_results)

entity_results

,Dialect,Entity Type,Gold,Correct,Recall
0,Barishal,ANI,49,17,0.346939
1,Barishal,COL,119,110,0.924370
2,Barishal,FOOD,260,165,0.634615
3,Barishal,LOC,313,204,0.651757
4,Barishal,OBJ,328,230,0.701220
5,Barishal,ORG,111,82,0.738739
6,Barishal,PER,35,33,0.942857
7,Barishal,REL,600,491,0.818333
8,Barishal,ROLE,83,47,0.566265
9,Chittagong,ANI,49,19,0.387755


In [122]:
error_counts = []

for dialect in char_error_results:

    boundary_errors = 0
    type_errors = 0
    missed_entities = 0

    for item in char_error_results[dialect]["predictions"]:

        gold = get_entities(item["true_tags"])
        pred = get_entities(item["pred_tags"])

        gold_spans = {
            (start, end)
            for start, end, _ in gold
        }

        pred_spans = {
            (start, end)
            for start, end, _ in pred
        }

        # Same boundary, wrong entity type
        for span in gold_spans & pred_spans:

            gold_type = [
                x[2] for x in gold
                if x[0] == span[0] and x[1] == span[1]
            ][0]

            pred_type = [
                x[2] for x in pred
                if x[0] == span[0] and x[1] == span[1]
            ][0]

            if gold_type != pred_type:
                type_errors += 1

        # Gold entities with incorrect boundaries
        for entity in gold:

            if entity[:2] not in pred_spans:
                missed_entities += 1

    error_counts.append({
        "Dialect": dialect,
        "Type Errors": type_errors,
        "Boundary/Missed Errors": missed_entities
    })

error_counts = pd.DataFrame(error_counts)

error_counts

,Dialect,Type Errors,Boundary/Missed Errors
0,Barishal,42,477
1,Chittagong,26,728
2,Mymensingh,42,223
3,Noakhali,63,426
4,Sylhet,75,357
